# Low-Light Image Enhancement: the pipeline, one step per cell

This notebook turns a dark photo into a clear one using **classical image processing only**.
There is no AI, no neural network and no training. Every step is a formula from an image processing course.

Each section below is **one stage of the pipeline**. Every section has a short explanation, the code that does that stage, and pictures of what it did.

| # | Stage | Topic |
|---|---|---|
| 1 | Illumination map | Retinex theory, edge-preserving filtering |
| 2 | Brighten | Gamma correction (on the light map) |
| 3 | Denoise | Noise estimation, non-local means |
| 4 | Local contrast | CLAHE (histogram processing) |
| 5 | Colour correction | Gray-world white balance, HSV |
| 6 | Sharpen | Unsharp masking |

Two bonus sections at the end compare the result with single-step methods, and show motion-blur removal with **Wiener deconvolution**.

**How to run:** *Runtime → Run all*, or press **Shift+Enter** on each cell from top to bottom.
To use your own photo, tick `USE_OWN_PHOTO` in Step 0.

In [ ]:
#@title Setup: imports and display helpers
import math
from dataclasses import dataclass, replace

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100


def to_float(img_u8):
    # uint8 [0, 255]  ->  float32 [0, 1]
    return img_u8.astype(np.float32) / 255.0


def to_u8(img_f):
    # float32 [0, 1]  ->  uint8 [0, 255], rounded rather than truncated
    return np.clip(img_f * 255.0 + 0.5, 0, 255).astype(np.uint8)


def show(images, titles=None, cols=None, size=5.0):
    # Show one or more BGR / grayscale images side by side.
    if not isinstance(images, (list, tuple)):
        images, titles = [images], [titles]
    titles = titles or [None] * len(images)
    cols = cols or len(images)
    rows = math.ceil(len(images) / cols)
    aspect = images[0].shape[0] / images[0].shape[1]
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows * aspect + 0.4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, im, t in zip(axes.flat, images, titles):
        if im.dtype != np.uint8:
            im = to_u8(im)
        if im.ndim == 2:
            ax.imshow(im, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        if t:
            ax.set_title(t, fontsize=11)
    plt.tight_layout()
    plt.show()


def heatmap(t):
    # Colour-mapped view of a [0, 1] map: dark purple = 0, yellow = 1.
    return cv2.applyColorMap(to_u8(t), cv2.COLORMAP_INFERNO)


print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

## Step 0: Load a dark photo

Three sample photos (600×400, shot in poor light) are embedded in the notebook.
Pick one with `SAMPLE`, or tick `USE_OWN_PHOTO` to upload your own.
Large photos are shrunk to 1600 px on the long side, as in the web app.

OpenCV stores colour images as **B, G, R** (not RGB), and the pipeline works on
**float values between 0 and 1**, so nothing overflows or rounds between steps.

In [ ]:
#@title Embedded sample photos (run this, no need to read it)
SAMPLES = {
    1: "...",
    2: "...",
    3: "...",
}
print("3 sample photos loaded")

In [ ]:
SAMPLE = 1             #@param [1, 2, 3]
USE_OWN_PHOTO = False  #@param {type:"boolean"}
MAX_SIDE = 1600

import base64
if USE_OWN_PHOTO:
    from google.colab import files
    uploaded = files.upload()
    data = next(iter(uploaded.values()))
else:
    data = base64.b64decode(SAMPLES[SAMPLE])

# imdecode also applies the phone's EXIF rotation tag
img_u8 = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
long_side = max(img_u8.shape[:2])
if long_side > MAX_SIDE:
    s = MAX_SIDE / long_side
    img_u8 = cv2.resize(img_u8, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

x_input = to_float(img_u8)
print(f"size: {img_u8.shape[1]}×{img_u8.shape[0]} px,  mean brightness: {x_input.mean():.3f}  (0 = black, 1 = white)")
show(img_u8, "Input photo")

## Settings and presets

Every adjustable number lives in one place. There are three presets:

- **`natural`** *(the web app's default)*: for everyday dark photos. Set by eye.
- **`detail`**: for text, signs and number plates. Lighter denoising (strong denoising smears small text), with stronger contrast and sharpening.
- **`benchmark`**: the values a grid search picked on 100 reference photos from the LOL dataset. Sharpening is switched off.

In [ ]:
@dataclass
class Params:
    # Stage 1: illumination map
    guide_radius: int = 30            # guided-filter window radius (px, at 600 px width)
    guide_eps: float = 1e-3           # guided-filter smoothness
    illum_floor: float = 0.02         # avoids dividing by ~0 in pitch-black pixels
    # Stage 2: brightening
    target_brightness: float = 0.40   # mean grey level the result should reach
    # Stage 3: denoising
    denoise_strength: float = 1.5     # multiplies the measured noise level
    denoise_dark_power: float = 1.0   # how strongly denoising favours dark regions
    # Stage 4: local contrast
    clahe_clip: float = 0.75
    clahe_tiles: int = 8
    # Stage 5: colour
    white_balance_strength: float = 0.5
    wb_max_gain: float | None = None
    saturation: float = 0.9
    # Stage 6: sharpening
    sharpen_amount: float = 0.4
    sharpen_sigma: float = 1.5


ALL = {"illumination", "brighten", "denoise", "contrast", "colour", "sharpen"}
PRESETS = {
    "benchmark": (Params(), ALL - {"sharpen"}),
    "natural":   (Params(denoise_strength=1.0, clahe_clip=1.5, saturation=1.1,
                         wb_max_gain=1.2, sharpen_amount=0.3), ALL),
    "detail":    (Params(denoise_strength=0.5, clahe_clip=2.5, saturation=1.0,
                         wb_max_gain=1.2, sharpen_amount=0.8, sharpen_sigma=1.0), ALL),
}

PRESET = "natural"  #@param ["natural", "detail", "benchmark"]
p, enabled = PRESETS[PRESET]
print("preset:", PRESET)
print(p)

## Stage 1: Illumination map: *where is the light?*

**Retinex model:** every photo is &nbsp; $I = R \times L$

- $R$ is **reflectance**: what the objects really look like.
- $L$ is **illumination**: how much light fell on each spot.

A dark photo has a good $R$ and a tiny $L$. If we can estimate $L$, we can **change the lighting without changing the objects.**

**1a. First guess: the brightest of B, G, R at each pixel.** A surface can't reflect more light than falls on it, so the brightest channel is the best lower bound on the light. (Using the max instead of the average means a red apple isn't mistaken for a dark spot.)

**1b. Smooth it with a guided filter.** Light changes slowly, but the raw map still contains all the texture of the scene. A normal blur would smear light across object edges, which later causes **halos**. The guided filter fits $q = a\cdot I + b$ in every window, with

$$a = \frac{\operatorname{var}}{\operatorname{var} + \varepsilon}$$

- In flat windows, $a \approx 0$: the map is averaged (smoothed).
- In windows across an edge, $a \approx 1$: the edge is kept.

**This stage only measures. The image itself is not changed yet.**

In [ ]:
def guided_filter(guide, src, r, eps):
    # Edge-preserving smoothing (He, Sun, Tang 2010). All means use box filters,
    # which cost the same per pixel for any window size.
    k = (2 * r + 1, 2 * r + 1)
    mean_I = cv2.boxFilter(guide, -1, k)
    mean_p = cv2.boxFilter(src, -1, k)
    corr_Ip = cv2.boxFilter(guide * src, -1, k)
    corr_II = cv2.boxFilter(guide * guide, -1, k)
    var_I = corr_II - mean_I * mean_I
    cov_Ip = corr_Ip - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return cv2.boxFilter(a, -1, k) * guide + cv2.boxFilter(b, -1, k)


def scaled_radius(r, img):
    # The radius was tuned on 600 px images; scale it to this photo's size.
    return max(1, round(r * max(img.shape[:2]) / 600))


t0 = x_input.max(axis=2)                                   # 1a: brightest channel
r = scaled_radius(p.guide_radius, x_input)
t = guided_filter(t0, t0, r, p.guide_eps)                   # 1b: edge-aware smoothing
t = np.clip(t, p.illum_floor, 1.0)                          # floor so we never divide by ~0

print(f"guided filter window: {2*r+1}×{2*r+1} px")
show([x_input, heatmap(t0), heatmap(t)],
     ["Input", "1a: brightest channel (raw)", "1b: illumination map (smoothed)"])

## Stage 2: Brighten: *gamma correction on the light, not the image*

We keep $R$ and replace the light $t$ with a brighter $t^\gamma$ ($\gamma < 1$):

$$\text{out} = R\cdot t^\gamma = \frac{\text{img}}{t}\cdot t^\gamma = \text{img}\cdot t^{\,\gamma-1}$$

So every pixel is multiplied by a **gain** $t^{\gamma-1}$:
- dark areas (small $t$) get a big boost
- bright areas (lamps, windows) get almost none, so they don't blow out
- **B, G and R get the same gain**, so colours keep their hue (ordinary gamma bends each channel separately and washes colours out)

**γ is chosen automatically by bisection** (repeatedly halving an interval) so the result's mean brightness hits `target_brightness`. Darker photos get a smaller γ, which means a stronger boost.

In [ ]:
def apply_gamma(img, t, gamma):
    gain = np.power(t, gamma - 1.0)
    return np.clip(img * gain[..., None], 0.0, 1.0)


# Search on a quarter-size copy: same answer, much faster.
small = cv2.resize(x_input, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_AREA)
t_small = cv2.resize(t, (small.shape[1], small.shape[0]), interpolation=cv2.INTER_AREA)

def mean_after(g):
    return cv2.cvtColor(apply_gamma(small, t_small, g), cv2.COLOR_BGR2GRAY).mean()

lo, hi = 0.05, 1.0                       # gamma = 1 means "leave unchanged"
if mean_after(hi) >= p.target_brightness:
    gamma = hi                           # already bright enough
else:
    for _ in range(20):                  # 20 halvings -> precision ~1e-6
        mid = (lo + hi) / 2
        if mean_after(mid) > p.target_brightness:
            lo = mid                     # too bright -> gamma can go up
        else:
            hi = mid                     # too dark   -> gamma must go down
    gamma = (lo + hi) / 2

x_bright = apply_gamma(x_input, t, gamma)
print(f"chosen gamma = {gamma:.3f}")
for tv in (0.05, 0.2, 0.5, 0.9):
    print(f"  light level {tv:<4} -> gain ×{tv ** (gamma - 1):.2f}")

# How the bisection sees the problem
gs = np.linspace(0.05, 1.0, 60)
plt.figure(figsize=(6, 3))
plt.plot(gs, [mean_after(float(g)) for g in gs], label="mean brightness after")
plt.axhline(p.target_brightness, color="orange", ls="--", label="target")
plt.axvline(gamma, color="green", ls=":", label=f"chosen γ = {gamma:.3f}")
plt.xlabel("γ"); plt.ylabel("mean brightness"); plt.legend(); plt.tight_layout(); plt.show()

plain_gamma = np.power(x_input, 1 / 2.2)     # ordinary gamma on the image, for comparison
show([x_input, plain_gamma, x_bright],
     ["Input", "Ordinary gamma (γ = 1/2.2) on the image", "Stage 2: gamma on the light map"])

## Stage 3: Denoise: *clean up the grain that brightening amplified*

Brightening multiplies the camera's noise by the same gain as the signal, so the shadows become grainy.

**3a. Measure the noise** (Immerkær 1996). The 3×3 mask below cancels smooth image content and responds mostly to pixel-level noise. A fixed formula turns its average response into the noise standard deviation σ.

**3b. Non-local means (NLM).** For each pixel, find patches elsewhere that *look similar* and average them. Edges and textures survive because only look-alike patches are mixed. The strength `h` is set from the measured σ, so a clean photo is barely touched and a grainy one is cleaned hard.

**3c. Blend by darkness.** Weight $w = (1-t)^{\text{power}}$. Areas that were dark (and had their noise boosted most) get the denoised version. Areas that were bright keep their original detail.

> On the reference dataset this stage gives **the biggest improvement**: SSIM goes from 0.49 to 0.73.

In [ ]:
def estimate_noise(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    resp = cv2.filter2D(gray, -1, mask)[1:-1, 1:-1]
    return float(np.sqrt(np.pi / 2) * np.abs(resp).mean() / 6.0)


if "denoise" in enabled:
    sigma = estimate_noise(x_bright)                                        # 3a
    h = max(1.0, sigma * 255.0 * p.denoise_strength)
    den = to_float(cv2.fastNlMeansDenoisingColored(to_u8(x_bright), None, h, h, 7, 21))   # 3b
    w = np.power(1.0 - t, p.denoise_dark_power)[..., None]                  # 3c
    x_denoised = w * den + (1.0 - w) * x_bright
    print(f"measured noise σ = {sigma:.4f}  (≈ {sigma*255:.1f} grey levels)  ->  NLM strength h = {h:.2f}")
else:
    x_denoised = x_bright

# Zoom into the centre so the grain is visible
H, W = x_bright.shape[:2]
crop = (slice(H // 4, H // 4 + H // 3), slice(W // 3, W // 3 + W // 3))
show([x_bright[crop], x_denoised[crop]], ["Before denoising (zoomed)", "After denoising (zoomed)"])
if "denoise" in enabled:
    show(heatmap(w[..., 0]), "Denoising weight: yellow = fully denoised, purple = kept as is", size=6)

## Stage 4: Local contrast: *CLAHE on lightness only*

Brightening with a smooth gain lifts the shadows but leaves the picture looking flat.
**CLAHE** (Contrast-Limited Adaptive Histogram Equalization) restores depth in 4 steps:

1. Split the image into tiles (8×8).
2. Give each tile its own histogram, **clipped** at a limit so noise in flat areas isn't blown up.
3. Equalize each tile.
4. Blend neighbouring tiles smoothly.

The **clip limit** sets how strong the effect is: low is gentle, high is punchy.

It runs only on the **L (lightness) channel of LAB colour space**, so contrast changes without shifting colours.

In [ ]:
if "contrast" in enabled:
    lab = cv2.cvtColor(to_u8(x_denoised), cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=p.clahe_clip, tileGridSize=(p.clahe_tiles, p.clahe_tiles))
    lab[..., 0] = clahe.apply(lab[..., 0])
    x_contrast = to_float(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR))
else:
    x_contrast = x_denoised

show([x_denoised, x_contrast], ["Before CLAHE", f"After CLAHE (clip {p.clahe_clip})"])

plt.figure(figsize=(7, 2.6))
for img, name in [(x_denoised, "before"), (x_contrast, "after")]:
    L = cv2.cvtColor(to_u8(img), cv2.COLOR_BGR2LAB)[..., 0]
    plt.hist(L.ravel(), bins=64, range=(0, 255), alpha=0.6, label=name)
plt.title("Histogram of lightness (L)"); plt.legend(); plt.tight_layout(); plt.show()

## Stage 5: Colour correction: *remove the colour cast*

**5a. Gray-world white balance.** Across a whole scene, colours usually average to neutral grey. If the average is yellowish, the *light* was yellow (light bulbs, street lamps). We scale B, G and R so their averages become equal.
It is applied at **partial strength** and the gain can be **capped**, because the assumption fails for scenes that really are one colour (a sunset would turn grey).

**5b. Saturation.** Convert to **HSV** and scale the S (saturation) channel to make colours richer or more muted.

In [ ]:
if "colour" in enabled:
    means = x_contrast.reshape(-1, 3).mean(axis=0)               # mean B, G, R
    gains = means.mean() / np.maximum(means, 1e-6)                # full gray-world gains
    gains = 1.0 + p.white_balance_strength * (gains - 1.0)        # partial strength
    if p.wb_max_gain:
        gains = np.clip(gains, 1.0 / p.wb_max_gain, p.wb_max_gain)
    balanced = np.clip(x_contrast * gains, 0.0, 1.0)

    hsv = cv2.cvtColor(balanced, cv2.COLOR_BGR2HSV)
    hsv[..., 1] = np.clip(hsv[..., 1] * p.saturation, 0.0, 1.0)
    x_colour = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    print("channel means  B={:.3f}  G={:.3f}  R={:.3f}".format(*means))
    print("gains applied  B×{:.3f}  G×{:.3f}  R×{:.3f}   then saturation ×{}".format(*gains, p.saturation))
else:
    x_colour = x_contrast

show([x_contrast, x_colour], ["Before colour correction", "After colour correction"])

## Stage 6: Sharpen: *unsharp masking*

$$\text{out} = \text{img} + \text{amount}\cdot(\text{img} - \text{blurred img})$$

`img − blurred` contains only the fine detail (edges). Adding some of it back makes edges crisper.
Sharpening comes **last** so it doesn't sharpen noise. The `benchmark` preset switches it off, because the
reference photos it was scored against are soft, and sharpening lowered the score there.

In [ ]:
if "sharpen" in enabled:
    blurred = cv2.GaussianBlur(x_colour, (0, 0), p.sharpen_sigma)
    x_final = np.clip(x_colour + p.sharpen_amount * (x_colour - blurred), 0.0, 1.0)
    detail = np.clip((x_colour - blurred) * 4 + 0.5, 0, 1)        # exaggerated for display
    show([x_colour[crop], detail[crop], x_final[crop]],
         ["Before (zoomed)", "Detail layer: img − blurred (×4)", f"After, amount {p.sharpen_amount} (zoomed)"])
else:
    x_final = x_colour
    print(f"Sharpening is off in the '{PRESET}' preset.")

## Result: all steps together

In [ ]:
show([img_u8, heatmap(t), x_bright, x_denoised, x_contrast, x_colour, x_final],
     ["Input", "1. Illumination map", f"2. Brighten (γ={gamma:.2f})", "3. Denoise",
      "4. Local contrast", "5. Colour", "6. Sharpen = final"], cols=4, size=4)

final_u8 = to_u8(x_final)
show([img_u8, final_u8], ["Before", "After"], size=7)
print(f"mean brightness: {x_input.mean():.3f} -> {x_final.mean():.3f}")

In [ ]:
#@title (Optional) Download the enhanced photo
cv2.imwrite("enhanced.jpg", final_u8)
try:
    from google.colab import files
    files.download("enhanced.jpg")
except ImportError:
    print("saved enhanced.jpg")

## Bonus A: How do single-step methods compare?

These are the standard textbook methods, each applied **on its own**:

- **Gamma correction:** one fixed curve on every channel.
- **Histogram equalization:** spreads the brightness over the full range.
- **CLAHE alone:** local contrast only, with no brightening.
- **Single-scale Retinex:** $\log I - \log(\text{blur}(I))$.

On the LOL test set (15 photo pairs, scores against the real well-lit photo):

| Method | PSNR (dB) ↑ | SSIM ↑ |
|---|---|---|
| CLAHE alone | 9.13 | 0.357 |
| Gamma alone | 13.23 | 0.590 |
| Histogram equalization | 14.26 | 0.425 |
| Single-scale Retinex | 15.57 | 0.597 |
| Homomorphic filtering | 16.15 | 0.383 |
| **Full pipeline (benchmark)** | **16.47** | **0.716** |

In [ ]:
def hist_eq(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.equalizeHist(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def clahe_only(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(2.0, (8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def ssr(img, sigma=80.0):
    f = img.astype(np.float32) + 1.0
    r = np.log1p(f) - np.log1p(cv2.GaussianBlur(f, (0, 0), sigma))
    out = np.zeros_like(r)
    for c in range(3):                                   # stretch each channel to 0..255
        ch = r[..., c]
        out[..., c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-6) * 255
    return out.astype(np.uint8)

gamma_only = to_u8(np.power(to_float(img_u8), 1 / 2.2))
show([img_u8, gamma_only, hist_eq(img_u8), clahe_only(img_u8), ssr(img_u8), final_u8],
     ["Input", "Gamma alone", "Histogram equalization", "CLAHE alone", "Single-scale Retinex", "Our pipeline"],
     cols=3, size=4.5)

## Bonus B: Removing motion blur with Wiener deconvolution

If a car (or the camera) moves during the shot, every point is smeared into a short line. That line is the
**point-spread function (PSF)**. Blur multiplies the image's spectrum by $H$ (the PSF's Fourier transform):

$$G = F\cdot H + \text{noise}$$

Dividing by $H$ would undo it, but $H$ is almost zero at some frequencies, and dividing noise by ≈0 explodes.
The **Wiener filter** stays safe at those frequencies:

$$\hat F = \frac{\overline{H}}{|H|^2 + \text{NSR}}\cdot G$$

NSR is the noise-to-signal ratio: small means strong deblurring, large means safe but softer.

Real number plates have unknown text, so this test **draws a plate with known text**, blurs it with known motion, adds noise,
and measures the recovery with **PSNR** (higher = closer to the original).
It also shows what happens when the blur direction is **guessed wrong**.

In [ ]:
def motion_psf(length, angle_deg):
    # A normalised straight line `length` px long at `angle_deg` (0 = horizontal).
    size = int(np.ceil(length)) | 1                 # odd size so the line is centred
    psf = np.zeros((size, size), np.float32)
    c = size // 2
    dx = np.cos(np.deg2rad(angle_deg)) * (length - 1) / 2
    dy = -np.sin(np.deg2rad(angle_deg)) * (length - 1) / 2
    cv2.line(psf, (round(c - dx), round(c - dy)), (round(c + dx), round(c + dy)), 1.0, 1, cv2.LINE_AA)
    return psf / psf.sum()


def wiener_deblur(img, psf, nsr):
    pad = psf.shape[0] * 2                         # mirror-pad so FFT wrap-around doesn't ring at the borders
    padded = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    h, w = padded.shape[:2]
    kernel = np.zeros((h, w), np.float32)
    kh, kw = psf.shape
    kernel[:kh, :kw] = psf
    kernel = np.roll(kernel, (-(kh // 2), -(kw // 2)), axis=(0, 1))   # PSF centre at (0, 0)
    H = np.fft.fft2(kernel)
    wiener = np.conj(H) / (np.abs(H) ** 2 + nsr)
    out = np.empty_like(padded)
    for ch in range(padded.shape[2]):
        out[..., ch] = np.real(np.fft.ifft2(np.fft.fft2(padded[..., ch]) * wiener))
    return np.clip(out[pad:-pad, pad:-pad], 0.0, 1.0)


def psnr(a, b):
    return 10 * np.log10(1.0 / np.mean((a - b) ** 2))


# A synthetic number plate with known text
plate = np.full((60, 240, 3), 235, np.uint8)
cv2.rectangle(plate, (2, 2), (237, 57), (0, 0, 0), 2)
cv2.putText(plate, "MH12 AB 3456", (10, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (20, 20, 20), 2, cv2.LINE_AA)
clean = to_float(cv2.copyMakeBorder(plate, 30, 30, 30, 30, cv2.BORDER_CONSTANT, value=(60, 60, 60)))

BLUR_LENGTH, BLUR_ANGLE, NSR = 15, 10, 0.005   #@param
psf = motion_psf(BLUR_LENGTH, BLUR_ANGLE)
blurred = cv2.filter2D(clean, -1, psf, borderType=cv2.BORDER_REFLECT)
blurred = np.clip(blurred + np.random.default_rng(0).normal(0, 0.01, blurred.shape), 0, 1).astype(np.float32)

fixed = wiener_deblur(blurred, psf, NSR)
wrong = wiener_deblur(blurred, motion_psf(BLUR_LENGTH, BLUR_ANGLE + 50), NSR)
inverse = wiener_deblur(blurred, psf, 1e-6)    # ~plain division by H

c = (slice(30, 90), slice(30, 270))
show([clean[c], blurred[c], inverse[c], wrong[c], fixed[c]],
     ["Original",
      f"Motion-blurred + noise   PSNR {psnr(clean, blurred):.1f} dB",
      f"Plain inverse filter (NSR≈0)   PSNR {psnr(clean, inverse):.1f} dB",
      f"Wiener, direction 50° WRONG   PSNR {psnr(clean, wrong):.1f} dB",
      f"Wiener, correct PSF   PSNR {psnr(clean, fixed):.1f} dB"], cols=1, size=7)

### What to take away

- **Every step is classical and explainable.** Retinex, gamma, filtering, histograms, colour spaces and the Fourier transform. There is no training data and no black box.
- **Order matters.** Brighten first, because other steps fail on near-black pixels. Denoise before contrast and colour, so they don't amplify grain. Sharpen last.
- **Adaptive, not fixed.** γ is found per photo, and the denoising strength follows the photo's measured noise.
- **Limits.** Brightening can't reveal detail the camera never recorded, and deblurring only helps when the blur length and direction are right.